In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

TRAIN_FN = "vol_dataset_train_20150102_20221230.csv"
VAL_FN = "vol_dataset_validation_20230103_20241231.csv"
TEST_FN = "vol_dataset_test_20250102_20251231.csv"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    _colab_root = Path("/content/drive/MyDrive/Colab Notebooks")
    _colab_sub = _colab_root / "data_splits"
    if (_colab_sub / TRAIN_FN).exists():
        DATA = _colab_sub
    else:
        DATA = _colab_root
except ModuleNotFoundError:
    _here = Path.cwd()
    DATA = _here / "data_splits"
    if not DATA.exists():
        DATA = _here / "207" / "207 project" / "Greed-and-Fear" / "data_splits"

FEAT = "trailing_vol_annual_decimel_20d_calculated"
TGT = "y_known_at_t"
FWD = "forward_vol_5d_annual_decimel_calculated"
# Target + V1/V2 features — same contract as Elastic_Net_Regression/elastic_net_vol_v1v2_vb_done_032726.ipynb
TARGET = FWD
CURRENT_STATE_FEATURES = [
    "y_known_at_t",
    "trailing_vol_annual_decimel_20d_calculated",
]
CURRENT_MACRO_MARKET_FEATURES = [
    "volume",
    "NYGOLDS",
    "OIL_WTI_S",
    "US_10Y_BOND_YLD",
    "US_3M_TB_YLD",
    "VIX",
]
FEATURES_V1 = list(CURRENT_STATE_FEATURES)
FEATURES_V2_EN = FEATURES_V1 + list(CURRENT_MACRO_MARKET_FEATURES)
# V1: same row filter as Elastic Net (all V2_EN inputs + forward target non-null).
ALIGN_COLS_V1 = FEATURES_V2_EN + [TARGET]
# V2: original LSTM — one input channel, predict y_known_at_t (dropna FEAT + TGT only).
FEATURES_V2_LSTM = [FEAT]
ALIGN_COLS_V2 = [FEAT, TGT]

def load_split(fn):
    p = DATA / fn
    df = pd.read_csv(p, low_memory=False)
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], format="mixed", errors="coerce")
    return df


def eda_one(name, df):
    n = len(df)
    na_f = df[FEAT].isna().sum()
    na_y = df[TGT].isna().sum()
    na_fwd = int(df[FWD].isna().sum()) if FWD in df.columns else None
    m_feat = df[FEAT].isna()
    m_tgt = df[TGT].isna()
    only_feat = int((m_feat & ~m_tgt).sum())
    only_tgt = int((~m_feat & m_tgt).sum())
    both = int((m_feat & m_tgt).sum())
    either = int((m_feat | m_tgt).sum())
    d = df.dropna(subset=[FEAT, TGT])


d_tr = load_split(TRAIN_FN)
d_va = load_split(VAL_FN)
d_te = load_split(TEST_FN)
eda_one(TRAIN_FN, d_tr)
eda_one(VAL_FN, d_va)
eda_one(TEST_FN, d_te)

Mounted at /content/drive


In [2]:
import tensorflow as tf
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

tf.keras.utils.set_random_seed(42)

SEQ = 5


def make_sequences(X, y, time_steps):
    xs, ys = [], []
    for i in range(len(X) - time_steps):
        xs.append(X.iloc[i : i + time_steps].values)
        ys.append(y.iloc[i + time_steps])
    return np.array(xs), np.array(ys)


def stack_sequences(df, feature_cols, align_cols, y_col):
    x_list, y_list = [], []
    for _, g in df.groupby("ticker", sort=False):
        g = g.sort_values("date")
        g = g.dropna(subset=align_cols)
        if len(g) <= SEQ:
            continue
        X = g[feature_cols]
        y = g[y_col]
        xa, ya = make_sequences(X, y, SEQ)
        if len(xa):
            x_list.append(xa)
            y_list.append(ya)
    if not x_list:
        n_f = len(feature_cols)
        return np.empty((0, SEQ, n_f)), np.array([])
    return np.vstack(x_list), np.concatenate(y_list)


def scale_splits(X_tr, X_va, X_te):
    sc = StandardScaler()
    nfeat = X_tr.shape[-1]
    X_tr_sc = sc.fit_transform(X_tr.reshape(-1, nfeat)).reshape(X_tr.shape)
    X_va_sc = sc.transform(X_va.reshape(-1, nfeat)).reshape(X_va.shape)
    X_te_sc = sc.transform(X_te.reshape(-1, nfeat)).reshape(X_te.shape)
    return sc, X_tr_sc, X_va_sc, X_te_sc


def build_lstm(shape_in):
    return tf.keras.Sequential([
        tf.keras.Input(shape=shape_in),
        tf.keras.layers.LSTM(50, activation="relu", return_sequences=True),
        tf.keras.layers.LSTM(50, activation="relu"),
        tf.keras.layers.Dense(1),
    ])


def train_lstm(name, X_tr, y_tr, X_va, y_va, X_te, y_te, epochs=30, batch_size=32):
    sc, X_tr_sc, X_va_sc, X_te_sc = scale_splits(X_tr, X_va, X_te)
    shape_in = (X_tr.shape[1], X_tr.shape[2])
    model = build_lstm(shape_in)
    model.compile(optimizer="adam", loss="mse", metrics=["mae", "mse"])
    print(f"=== {name} (n_features={shape_in[1]}) ===")
    model.fit(
        X_tr_sc,
        y_tr,
        validation_data=(X_va_sc, y_va),
        epochs=epochs,
        batch_size=batch_size,
        verbose=1,
        shuffle=False,
    )
    pred = np.clip(model.predict(X_te_sc, verbose=0).flatten(), 0, None)
    mae = mean_absolute_error(y_te, pred)
    rmse = float(np.sqrt(mean_squared_error(y_te, pred)))
    print(f"{name} test mae={mae} rmse={rmse} seq_shapes", X_tr.shape, X_va.shape, X_te.shape)
    return model, sc, X_te_sc, pred, mae, rmse


X_tr_v1, y_tr_v1 = stack_sequences(d_tr, FEATURES_V1, ALIGN_COLS_V1, TARGET)
X_va_v1, y_va_v1 = stack_sequences(d_va, FEATURES_V1, ALIGN_COLS_V1, TARGET)
X_te_v1, y_te_v1 = stack_sequences(d_te, FEATURES_V1, ALIGN_COLS_V1, TARGET)

X_tr_v2, y_tr_v2 = stack_sequences(d_tr, FEATURES_V2_LSTM, ALIGN_COLS_V2, TGT)
X_va_v2, y_va_v2 = stack_sequences(d_va, FEATURES_V2_LSTM, ALIGN_COLS_V2, TGT)
X_te_v2, y_te_v2 = stack_sequences(d_te, FEATURES_V2_LSTM, ALIGN_COLS_V2, TGT)

model_v1, sc_v1, X_te_sc_v1, pred_v1, mae_v1, rmse_v1 = train_lstm(
    "LSTM V1: 2 vol inputs → forward 5d vol)", X_tr_v1, y_tr_v1, X_va_v1, y_va_v1, X_te_v1, y_te_v1
)
model_v2, sc_v2, X_te_sc_v2, pred_v2, mae_v2, rmse_v2 = train_lstm(
    "LSTM V2 (original: trailing 20d vol only → y_known_at_t)", X_tr_v2, y_tr_v2, X_va_v2, y_va_v2, X_te_v2, y_te_v2
)
print("V1 target:", TARGET, "n_features:", len(FEATURES_V1), "| V2 target:", TGT, "n_features:", len(FEATURES_V2_LSTM))

=== LSTM V1: 2 vol inputs → forward 5d vol) (n_features=2) ===
Epoch 1/30
2143/2143 ━━━━━━━━━━━━━━━━━━━━ 22s 9ms/step - loss: 0.0171 - mae: 0.0818 - mse: 0.0171 - val_loss: 0.0084 - val_mae: 0.0700 - val_mse: 0.0084
Epoch 2/30
2143/2143 ━━━━━━━━━━━━━━━━━━━━ 18s 8ms/step - loss: 0.0160 - mae: 0.0800 - mse: 0.0160 - val_loss: 0.0082 - val_mae: 0.0687 - val_mse: 0.0082
Epoch 3/30
2143/2143 ━━━━━━━━━━━━━━━━━━━━ 23s 10ms/step - loss: 0.0158 - mae: 0.0798 - mse: 0.0158 - val_loss: 0.0082 - val_mae: 0.0682 - val_mse: 0.0082
Epoch 4/30
2143/2143 ━━━━━━━━━━━━━━━━━━━━ 18s 8ms/step - loss: 0.0155 - mae: 0.0795 - mse: 0.0155 - val_loss: 0.0081 - val_mae: 0.0677 - val_mse: 0.0081
Epoch 5/30
2143/2143 ━━━━━━━━━━━━━━━━━━━━ 18s 8ms/step - loss: 0.0154 - mae: 0.0794 - mse: 0.0154 - val_loss: 0.0082 - val_mae: 0.0677 - val_mse: 0.0082
Epoch 6/30
2143/2143 ━━━━━━━━━━━━━━━━━━━━ 18s 8ms/step - loss: 0.0153 - mae: 0.0793 - mse: 0.0153 - val_loss: 0.0082 - val_mae: 0.0673 - val_mse: 0.0082
Epoch 7/30
2143/21

In [3]:
from sklearn.metrics import r2_score

print("LSTM V1 (2 inputs → forward 5d vol)")
model_v1.summary()
ev1 = model_v1.evaluate(X_te_sc_v1, y_te_v1, verbose=0, return_dict=True)
print("test_evaluate", {k: float(v) for k, v in ev1.items()})
print("r2_score", float(r2_score(y_te_v1, pred_v1)))
print("mae_rmse_sklearn (pred clipped >=0)", mae_v1, rmse_v1)

print("LSTM V2 (1 input → y_known_at_t)")
model_v2.summary()
ev2 = model_v2.evaluate(X_te_sc_v2, y_te_v2, verbose=0, return_dict=True)
print("test_evaluate", {k: float(v) for k, v in ev2.items()})
print("r2_score", float(r2_score(y_te_v2, pred_v2)))
print("mae_rmse_sklearn (pred clipped >=0)", mae_v2, rmse_v2)

LSTM V1 (2 inputs → forward 5d vol)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 5, 50)          │        10,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 50)             │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 92,555 (361.55 KB)

 Trainable params: 30,851 (120.51 KB)

...

 Total params: 91,955 (359.20 KB)

 Trainable params: 30,651 (119.73 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 61,304 (239.47 KB)

test_evaluate {'loss': 0.005791529547423124, 'mae': 0.050922900438308716, 'mse': 0.005791529547423124}
r2_score 0.7418352898725399
mae_rmse_sklearn (pred clipped >=0) 0.05092288777382737 0.07610211108204887
